# SPICE Self-Play Pipeline
This notebook identically mimics the logic of the `train_spice_selfplay.py` script.

In [ ]:
!pip install -U pip setuptools wheel
!pip install unsloth trl pydantic pyyaml "openenv-core>=0.2.0"
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-hn4w7ddz/unsloth_161620f92a5f45b6af227ba1f89e933c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-hn4w7ddz/unsloth_161620f92a5f45b6af227ba1f89e933c
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Setup Local Environment

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "spicy-attacker"
WORKDIR = Path("/content/MetaHackathon-R2")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
else:
    os.chdir(WORKDIR)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)

import sys
if str(WORKDIR / "src") not in sys.path: sys.path.append(str(WORKDIR / "src"))
if str(WORKDIR / "scripts") not in sys.path: sys.path.append(str(WORKDIR / "scripts"))
os.chdir(WORKDIR)

## 2. Load Parent Scenarios & Model

In [ ]:
from pathlib import Path
import random
from train_spice_selfplay import load_parent_specs, ensure_lora_is_trainable
from oncallenv.core.types import ScenarioSpec

parent_specs = load_parent_specs(Path("scenarios_seed"), Path("curriculum_results/buffer.json"), 120, 20260424)
print(f"Loaded {len(parent_specs)} parent scenarios")

from unsloth import FastLanguageModel, PatchFastRL
try: PatchFastRL("GRPO", FastLanguageModel)
except: pass

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=1536,
    load_in_4bit=True,
    fast_inference=False,
)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.train()
ensure_lora_is_trainable(model)

Loaded 120 parent scenarios
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_12823/67398329.py:9: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.

model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


{'total': 907081216, 'trainable': 18464768, 'lora_tensors': 392}

## 3. SPICE Generation Phase

In [ ]:
from tqdm.auto import tqdm
from train_spice_selfplay import selfplay_iteration

all_attacker_data, all_defender_data = [], []
selfplay_iterations = 2  # Smoke test size (was 20)
batch_size = 4           # Smoke test size (was 8)
selfplay_group_size = 2  # Smoke test size (was 6)
temperature = 0.8
challenger_penalty = -0.1
rng = random.Random(20260424)

for iteration in tqdm(range(selfplay_iterations), desc="SPICE Generation Phase"):
    batch = rng.sample(parent_specs, min(batch_size, len(parent_specs)))
    result = selfplay_iteration(
        model=model, tokenizer=tokenizer, parent_specs=batch,
        group_size=selfplay_group_size, generation=iteration,
        temperature=temperature, max_attacker_tokens=200, max_defender_tokens=256,
        challenger_penalty=challenger_penalty
    )
    all_attacker_data.extend(result["attacker_rows"])
    all_defender_data.extend(result["defender_rows"])


SPICE Generation Phase:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

## 4. Build Dataset

In [ ]:
from spice_defender import build_defender_prompt
from llm_attacker import build_attacker_prompt
from oncallenv.simulation.scenario_compiler import compile_scenario

train_rows = []
for row in all_attacker_data:
    spec = next((s for s in parent_specs if s.task_id == row["parent"]), parent_specs[0])
    train_rows.append({
        "prompt": [{"role": "user", "content": build_attacker_prompt(spec)}],
        "role": "attacker",
        "parent_task_id": row["parent"],
        "task_id": row.get("child_task_id", ""),
        "spec": None,
        "root_service": "",
        "root_category": "",
    })

for row in all_defender_data:
    spec_dict = row.get("spec")
    spec = ScenarioSpec.model_validate(spec_dict) if spec_dict else next((s for s in parent_specs if s.task_id == row["task_id"]), None)
    if not spec: continue
    graph = compile_scenario(spec)
    train_rows.append({
        "prompt": [{"role": "user", "content": build_defender_prompt(spec)}],
        "role": "defender",
        "parent_task_id": "",
        "task_id": spec.task_id,
        "spec": spec.model_dump(),
        "root_service": graph.root_cause_service,
        "root_category": graph.root_cause_category,
    })

rng.shuffle(train_rows)
print(f"Combined training dataset: {len(train_rows)} rows")

Combined training dataset: 32 rows


## 5. Quick Test Run & Full Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

test_rows = train_rows[:4]
while len(test_rows) < 4: test_rows.extend(test_rows)
test_dataset = Dataset.from_list(test_rows[:4])
full_dataset = Dataset.from_list(train_rows)

def _get(kwargs, key, idx):
    v = kwargs.get(key)
    if v is None: return None
    return v[idx] if isinstance(v, list) else v

def attacker_reward(completions, **kwargs):
    from llm_attacker import parse_attacker_actions
    from spice_defender import extract_completion_text, defender_rollout_reward, build_defender_prompt
    from train_spice_selfplay import generate_text
    import math

    rewards = []
    comps_list = list(completions)
    for idx, completion in enumerate(comps_list):
        r_val = _get(kwargs, "role", idx)
        if r_val != "attacker":
            rewards.append(0.0)
            continue
        text = extract_completion_text(completion)
        pid = _get(kwargs, "parent_task_id", idx)
        parent = next(s for s in parent_specs if s.task_id == pid)
        spec, is_valid, _ = parse_attacker_actions(text, parent)
        if not is_valid or spec is None:
            rewards.append(challenger_penalty * 0.5 if "set_field" in text.lower() or "<actions>" in text.lower() else challenger_penalty)
            continue
        try: compile_scenario(spec)
        except Exception:
            rewards.append(challenger_penalty * 0.5 if "set_field" in text.lower() or "<actions>" in text.lower() else challenger_penalty)
            continue

        # Run live Defender rollouts against the newly generated Attacker spec
        d_prompt = build_defender_prompt(spec)
        # We use 2 defender rollouts during the GRPO phase to save time while still getting a variance signal
        d_comps = generate_text(model, tokenizer, [d_prompt], max_new_tokens=256, temperature=0.8, num_return=2)

        d_passes = []
        for d_comp in d_comps:
            try:
                r = defender_rollout_reward(spec, d_comp)
                d_passes.append(1.0 if r >= 0.3 else 0.0)
            except Exception:
                d_passes.append(0.0)

        p = sum(d_passes) / len(d_passes)
        tau = 0.1
        gaussian_reward = math.exp(-((p - 0.5) ** 2) / (2 * tau))
        rewards.append(gaussian_reward)

    return rewards

def defender_reward(completions, **kwargs):
    from spice_defender import extract_completion_text, defender_rollout_reward
    from llm_attacker import normalize_defender_reward
    rewards = []
    comps_list = list(completions)
    for idx, completion in enumerate(comps_list):
        r_val = _get(kwargs, "role", idx)
        if r_val != "defender":
            rewards.append(0.0)
            continue
        text = extract_completion_text(completion)
        spec_dict = _get(kwargs, "spec", idx)
        spec = ScenarioSpec.model_validate(spec_dict)
        try: r = defender_rollout_reward(spec, text)
        except Exception: r = -0.25
        rewards.append(normalize_defender_reward(r))
    return rewards

# To run the full dataset instead of the test slice:
# 1. Change train_dataset=test_dataset to train_dataset=full_dataset
# 2. Change max_steps=2 to max_steps=600

print("Running 2-step test...")
config = GRPOConfig(output_dir="spice_outputs_test", learning_rate=5e-6, per_device_train_batch_size=2,
                    gradient_accumulation_steps=1, num_generations=2, max_prompt_length=1024,
                    max_completion_length=128, max_steps=2, beta=0.0, loss_type="dr_grpo")

trainer = GRPOTrainer(model=model, reward_funcs=[attacker_reward, defender_reward], args=config, train_dataset=test_dataset)
trainer.train()
print("Test run successful!")

Running 2-step test...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151654}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 1 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. 

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / attacker_reward / mean,rewards / attacker_reward / std,rewards / defender_reward / mean,rewards / defender_reward / std
1,-0.237457,0.093252,0.273300,65.000000,22.000000,108.000000,0.000000,65.000000,22.000000,108.000000,0.000000,0.093252,0.273300,0.000000,0.000000
2,0.000000,0.286505,0.000000,18.500000,17.000000,20.000000,0.000000,18.500000,17.000000,20.000000,0.000000,0.286505,0.000000,0.000000,0.000000


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

Test run successful!
